# D2 — What emissions are present, missing and changing?

This notebook counts MEGAN emissions in interactive CTRL and LCC members. It answers two different questions: **how much of the baseline emissions lies outside ISOP/MTERP**, and **how much each group changes between simulations**.

It does not calculate ozone production, methane lifetime, actual OH reactivity or radiative forcing. Mass and carbon are budgets. The `koh` weighting multiplies emitted moles by kOH(species)/kOH(CO) at T_REF and 101325 Pa: it is an emissions-based index, not ΣkOH×concentration in s⁻¹.

The compound → CAM-species mapping, molecular formulas and OH rate constants below follow the MEGAN specifier and `chem_mech.in`. Check them, and the CLM flux convention (per land area vs per grid-cell area), if a result looks surprising.

**Use:** Python 3.10+; numpy, pandas, xarray, scipy, matplotlib ≥3.7, dask, cftime and a NetCDF backend. Run top to bottom. Paths and case names are set in step 1. Everything else is defined in this notebook; no D1 output and no `boreal_forest_expansion` import is needed.


In [ ]:
from __future__ import annotations
from pathlib import Path
from dataclasses import dataclass
from typing import Iterable
import glob, logging, math, re
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.ticker import MaxNLocator
from scipy import stats as sps
from IPython.display import display
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
log = logging.getLogger("diagnostics")
pd.set_option("display.max_columns", 30, "display.width", 160)
plt.rcParams.update({"font.size": 10, "figure.dpi": 110})

@dataclass
class EmissionContrast:
    """minuend - subtrahend, both CLM h0 globs."""
    name: str
    minuend: str
    subtrahend: str
    label: str = ""
    note: str = ""


## 1. Set input files and output folders


In [ ]:
ARCHIVE = Path("/cluster/work/users/adelez/archive")
CASES = {   # production cases (CLAUDE.md § Simulations)
    "CTRL_PD":  "NF2000norbc_tropstratchem_nudg_ctrl_f19_f19-20260925",
    "LCC_PD":   "NF2000norbc_tropstratchem_nudg_lcc_f19_f19-20261005",
    "CTRL_FUT": "NF2100ssp585norbc_tropstratchem_nudg_ctrl_f19_f19-20260925",
    "LCC_FUT":  "NF2100ssp585norbc_tropstratchem_nudg_lcc_f19_f19-20261005",
}
CLM = {a: str(ARCHIVE / c / "lnd" / "hist" / f"{c}.clm2.h0.*.nc") for a, c in CASES.items()}
CAM_CTRL_PD = str(ARCHIVE / CASES["CTRL_PD"] / "atm" / "hist" / f"{CASES['CTRL_PD']}.cam.h0.*.nc")  # cross-check only

REPO = Path("/cluster/home/adelez/BOREAL-FOREST-EXPANSION")
MODIFIED_CELLS = REPO / "diagnostics/check-simulation-equilibrium/lcc-modified-cells/modified_cells.nc"
if not MODIFIED_CELLS.exists():
    MODIFIED_CELLS = MODIFIED_CELLS.with_suffix(".csv")

SPINUP_MONTHS = 12
T_REF = 298.0            # K, for k_OH
FIG_DIR = REPO / "diagnostics/other-BVOC-emission-magnitude/D2_megan_budget/figures"
TABLE_DIR = REPO / "diagnostics/other-BVOC-emission-magnitude/D2_megan_budget/tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

MEMBERS_Q1 = {"CTRL_PD": CLM["CTRL_PD"], "CTRL_FUT": CLM["CTRL_FUT"]}
CONTRASTS = [
    EmissionContrast("LCC-CTRL_PD", CLM["LCC_PD"], CLM["CTRL_PD"], note="LCC, PD"),
    EmissionContrast("LCC-CTRL_FUT", CLM["LCC_FUT"], CLM["CTRL_FUT"], note="LCC only, 2100 climate"),
    EmissionContrast("LCC_FUT-CTRL_PD", CLM["LCC_FUT"], CLM["CTRL_PD"],
                        note="LCC + climate change: the emission change inside BVOC_FUT"),
]
CONTRASTS.append(EmissionContrast("CTRL_FUT-CTRL_PD", CLM["CTRL_FUT"], CLM["CTRL_PD"], note="Background environmental change"))
RUN_CAM_CROSSCHECK = False  # Enable after verifying CAM MEG_* units and availability.


## Notebook-local helpers
These definitions live here so the notebook needs no project imports or generator scripts. The actual workflow follows below in numbered steps. Ordinary Python functions keep repeated calculations consistent; none need to be installed in `src`.


### Read monthly files and place them in the correct month


In [ ]:
log = logging.getLogger(__name__)


WEIGHT_VARS = ("gw", "GRIDAREA", "LANDFRAC", "area", "landfrac")


COMPONENT_TAGS = (".cam.", ".clm2.")


def _case_of(path: str) -> str:
    """Case name = file name up to '.cam.' or '.clm2.' (e.g. NF2000..._ctrl_f19_f19-20260925)."""
    name = Path(path).name
    for tag in COMPONENT_TAGS:
        if tag in name:
            return name.split(tag)[0]
    return name


def find_files(pattern: str | Path) -> list[str]:
    """Expand a glob and refuse to mix two different cases.

    A wildcard in the case date (e.g. '...lcc_f19_f19-*') is convenient, but if two
    runs with different dates exist the files would be silently merged. That is an
    error here, not a warning.
    """
    files = sorted(glob.glob(str(pattern)))
    if not files:
        raise FileNotFoundError(f"No files match {pattern}")
    cases = sorted({_case_of(f) for f in files})
    if len(cases) > 1:
        raise ValueError(
            f"Pattern {pattern} matches files from {len(cases)} cases: {cases}. "
            "Put the full case name (with date) in the pattern."
        )
    return files


def _bounds_name(ds: xr.Dataset) -> str | None:
    for name in ("time_bnds", "time_bounds"):
        if name in ds.variables:
            return name
    return None


def fix_time(ds: xr.Dataset) -> xr.Dataset:
    """Stamp each monthly mean at the midpoint of its averaging interval.

    CAM writes monthly means with time = end of the interval, i.e. the January mean
    is stamped 1 February (a: verified in the earlier postprocessing work). Grouping
    by year/month on the raw stamp therefore shifts everything by one month.
    Also stores the interval length in days as `days_in_interval` (used as weight).
    """
    bname = _bounds_name(ds)
    if bname is None:
        raise ValueError(
            "No time bounds in the file: cannot place monthly means correctly. "
            "Open raw h0 files, or keep time_bnds when selecting variables."
        )
    bnds = ds[bname].load()
    bdim = [d for d in bnds.dims if d != "time"][0]
    start, end = bnds.isel({bdim: 0}).values, bnds.isel({bdim: 1}).values
    length = end - start                      # timedelta64 or datetime.timedelta objects
    mid = start + length / 2
    if np.issubdtype(np.asarray(length).dtype, np.timedelta64):
        days = length / np.timedelta64(1, "D")
    else:
        days = np.array([t.total_seconds() / 86400.0 for t in length])
    out = ds.assign_coords(time=("time", mid, ds.time.attrs))
    out["days_in_interval"] = ("time", np.asarray(days, dtype=float))
    return out


def drop_spinup(ds: xr.Dataset, months: int = 12) -> xr.Dataset:
    """Remove the first `months` monthly means (decision: 12 months of spin-up)."""
    if months <= 0:
        return ds
    if ds.sizes["time"] <= months:
        raise ValueError(f"Only {ds.sizes['time']} months in file, cannot drop {months} of spin-up")
    return ds.isel(time=slice(months, None))


def open_cam_monthly(
    pattern: str | Path,
    variables: Iterable[str],
    spinup_months: int = 12,
    strict: bool = True,
) -> xr.Dataset:
    """Open monthly CAM or CLM output for one case: selected variables, fixed time, no spin-up.

    Works for CLM h0 too (<case>.clm2.h0.*.nc): time bounds are found under either
    name, and `area`/`landfrac` are kept for land weights.

    Parameters
    ----------
    pattern
        Glob matching the h0 files of ONE case (see `find_files`).
    variables
        Data variables to read.
    spinup_months
        Number of leading months to drop.
    strict
        True: a missing variable raises KeyError. False: missing variables are listed in
        ds.attrs['missing'] and skipped.
    """
    files = find_files(pattern)
    variables = list(variables)
    keep = set(variables) | set(WEIGHT_VARS) | {"time_bnds", "time_bounds"}

    def _select(ds: xr.Dataset) -> xr.Dataset:
        absent = set(variables) - set(ds.data_vars)
        if absent and strict:
            raise KeyError(f"Missing fields in {ds.encoding.get('source', 'input')}: {sorted(absent)}")
        present = [v for v in ds.data_vars if v in keep]
        return ds[present]

    try:  # xarray >= 2025.01
        time_kw = {"decode_times": xr.coders.CFDatetimeCoder(use_cftime=True)}
    except AttributeError:  # older xarray
        time_kw = {"use_cftime": True}
    ds = xr.open_mfdataset(
        files,
        combine="by_coords",
        preprocess=_select,
        data_vars="minimal",
        coords="minimal",
        compat="override",
        parallel=False,
        **time_kw,
    )
    missing = [v for v in variables if v not in ds.data_vars]
    if missing and strict:
        raise KeyError(f"{_case_of(files[0])}: missing {missing}. Present: {sorted(ds.data_vars)}")
    if missing:
        log.warning("%s: %d requested variables missing: %s", _case_of(files[0]), len(missing), missing)

    ds = fix_time(ds)
    keys = ds.time.dt.year.values * 12 + ds.time.dt.month.values
    if np.any(np.diff(keys) != 1):
        raise ValueError("Duplicate or missing months; check overlapping raw/prepared files.")
    # CLM's first month starts one model step (30 min) early, so allow ~1 h of slack.
    if not np.allclose(ds.days_in_interval, ds.time.dt.days_in_month, rtol=0, atol=0.05):
        raise ValueError("Intervals are not full monthly means.")
    ds = drop_spinup(ds, spinup_months)
    ds.attrs["missing"] = ",".join(missing)
    ds.attrs["case"] = _case_of(files[0])
    ds.attrs["n_files"] = len(files)
    log.info("%s: %d files, %d months after spin-up", ds.attrs["case"], len(files), ds.sizes["time"])
    return ds


def annual_mean(ds: xr.Dataset | xr.DataArray, days: xr.DataArray) -> xr.Dataset | xr.DataArray:
    """Annual means weighted by month length; incomplete years are dropped (with a log line).

    Every variable in `ds` must have a time dimension (pass only the fields to average,
    not gw or other static fields, which would otherwise be broadcast over time).
    """
    if isinstance(ds, xr.Dataset):
        static = [v for v in ds.data_vars if "time" not in ds[v].dims]
        if static:
            raise ValueError(f"annual_mean: variables without time dimension {static}; drop them first")
    year = ds["time"].dt.year
    counts = days.groupby(year).count()
    complete = counts["year"].where(counts == 12, drop=True).values
    incomplete = sorted(set(counts["year"].values) - set(complete))
    if incomplete:
        log.warning("Dropping incomplete years %s", incomplete)
    w = days.groupby(year) / days.groupby(year).sum()
    out = (ds * w).groupby(year).sum("time", skipna=False)
    return out.sel(year=complete)


### Select regions and calculate area weights


In [ ]:
log = logging.getLogger(__name__)


def area_weights(ds: xr.Dataset) -> xr.DataArray:
    """2-D (lat, lon) weights proportional to cell area, normalised to sum 1."""
    if "gw" in ds:
        w = ds["gw"]
        if "time" in w.dims:
            w = w.isel(time=0, drop=True)
        w = w.broadcast_like(ds["lon"]).transpose("lat", "lon")
        source = "gw"
    elif "GRIDAREA" in ds:
        w = ds["GRIDAREA"]
        if "time" in w.dims:
            w = w.isel(time=0, drop=True)
        source = "GRIDAREA"
    else:
        w = np.cos(np.deg2rad(ds["lat"])).broadcast_like(ds["lon"]).transpose("lat", "lon")
        source = "cos(lat)"
    w = (w / w.sum()).load()
    w.name = "area_weight"
    w.attrs["source"] = source
    return w


def band_mask(ds: xr.Dataset, lat_min: float, lat_max: float) -> xr.DataArray:
    """True for cells whose centre lies in [lat_min, lat_max]."""
    lat2d = ds["lat"].broadcast_like(ds["lon"]).transpose("lat", "lon")
    return ((lat2d >= lat_min) & (lat2d <= lat_max)).rename("mask")


def load_modified_cells(path: str | Path, ds: xr.Dataset, tol_deg: float = 1e-3) -> xr.DataArray:
    """Mask of land-cover-modified cells on the grid of `ds`.

    Accepts the cache written by build_modified_cells (modified_cells.nc, variable
    `modified`) or its CSV twin (columns lat, lon_360). Cells are matched by
    coordinate within `tol_deg`; any unmatched cell raises (never a nearest-neighbour
    regrid, consistent with equilibrium.modified_mask).
    """
    path = Path(path)
    if path.suffix == ".nc":
        with xr.open_dataset(path) as m:
            m = m.load()
        lat2d, lon2d = xr.broadcast(m["lat"], m["lon"])
        sel = m["modified"].fillna(0).transpose("lat", "lon").values.astype(bool)
        lats, lons = lat2d.values[sel], lon2d.values[sel] % 360
    elif path.suffix == ".csv":
        t = pd.read_csv(path)
        lats, lons = t["lat"].to_numpy(), t["lon_360"].to_numpy() % 360
    else:
        raise ValueError(f"Expected .nc or .csv, got {path}")

    if len(lats) == 0:
        raise ValueError("The modified-cell mask is empty.")
    glat, glon = ds["lat"].values, ds["lon"].values % 360
    i = np.abs(glat[None, :] - lats[:, None]).argmin(axis=1)
    dlon = np.abs(glon[None, :] - lons[:, None])
    dlon = np.minimum(dlon, 360 - dlon)
    j = dlon.argmin(axis=1)
    err = np.maximum(np.abs(glat[i] - lats), dlon[np.arange(len(j)), j])
    if np.any(err > tol_deg):
        raise ValueError(f"{(err > tol_deg).sum()} modified cells do not match the grid within {tol_deg} deg")

    mask = np.zeros((glat.size, glon.size), dtype=bool)
    mask[i, j] = True
    log.info("Modified cells: %d from %s", mask.sum(), path.name)
    return xr.DataArray(mask, coords={"lat": ds["lat"], "lon": ds["lon"]}, dims=("lat", "lon"), name="modified")


def default_regions(ds: xr.Dataset, modified: xr.DataArray) -> dict[str, xr.DataArray]:
    """Regions for the far-field check.

    tropics, SH_extratropics: far from the land-cover change (all modified cells are
    north of 46 N (a, modified_cells.csv)); modified_cells: where the BVOC signal is
    expected, shown for scale; global: everything.
    """
    regions = {
        "modified_cells": modified,
        "tropics": band_mask(ds, -30, 30),
        "SH_extratropics": band_mask(ds, -90, -30.0001),
        "global": xr.ones_like(modified, dtype=bool),
    }
    for name in ("tropics", "SH_extratropics"):
        if bool((regions[name] & modified).any()):
            raise ValueError(f"Region {name} overlaps modified cells: not a far-field region")
    return regions


def area_fraction(weights: xr.DataArray, mask: xr.DataArray) -> float:
    return float(weights.where(mask).sum())


def regional_mean(da: xr.DataArray, weights: xr.DataArray, mask: xr.DataArray) -> xr.DataArray:
    """Area-weighted mean over the cells where mask is True (keeps other dims, e.g. year)."""
    w = weights.where(mask, 0.0)
    return da.weighted(w).mean(("lat", "lon"))


### Summarise paired annual differences


In [ ]:
def lag1_autocorr(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size < 3 or np.ptp(x) == 0:
        return np.nan
    a = x - x.mean()
    return float(np.sum(a[1:] * a[:-1]) / np.sum(a * a))


def paired_summary(d: xr.DataArray | np.ndarray, conf: float = 0.95, n_boot: int = 10000,
                   seed: int = 0) -> dict:
    """Mean of yearly differences with t-based and bootstrap intervals.

    Returns mean, sd, n, r1, n_eff, se (with n_eff), ci_lo/ci_hi (t, n_eff - 1 dof),
    p (two-sided, H0: mean = 0), boot_lo/boot_hi (percentile bootstrap over years;
    with only ~9 years it is narrower than the t interval and is shown for reference).
    """
    x = np.asarray(d, dtype=float).ravel()
    x = x[np.isfinite(x)]
    n = x.size
    out = {"mean": np.nan, "sd": np.nan, "n": n, "r1": np.nan, "n_eff": np.nan, "se": np.nan,
           "ci_lo": np.nan, "ci_hi": np.nan, "p": np.nan, "boot_lo": np.nan, "boot_hi": np.nan}
    if n < 3:
        return out
    mean, sd = x.mean(), x.std(ddof=1)
    r1 = lag1_autocorr(x)
    r1_pos = max(r1, 0.0) if np.isfinite(r1) else 0.0
    n_eff = max(n * (1 - r1_pos) / (1 + r1_pos), 2.0)
    se = sd / np.sqrt(n_eff)
    tcrit = sps.t.ppf(0.5 + conf / 2, df=n_eff - 1)
    p = 2 * sps.t.sf(abs(mean) / se, df=n_eff - 1) if se > 0 else (0.0 if mean != 0 else 1.0)
    rng = np.random.default_rng(seed)
    boot = x[rng.integers(0, n, size=(n_boot, n))].mean(axis=1)
    lo, hi = np.percentile(boot, [50 * (1 - conf), 50 * (1 + conf)])
    out.update(mean=mean, sd=sd, r1=r1, n_eff=n_eff, se=se, ci_lo=mean - tcrit * se,
               ci_hi=mean + tcrit * se, p=p, boot_lo=lo, boot_hi=hi)
    return out


def cellwise_pvalue(d: xr.DataArray, dim: str = "year") -> xr.DataArray:
    """Two-sided one-sample t-test of the yearly differences in every grid cell."""
    def _p(x):
        with np.errstate(invalid="ignore", divide="ignore"):
            return sps.ttest_1samp(x, 0.0, axis=-1, nan_policy="omit").pvalue
    p = xr.apply_ufunc(_p, d, input_core_dims=[[dim]], dask="allowed")
    return p.rename("p")


def fdr_mask(p: xr.DataArray, alpha: float = 0.10, within: xr.DataArray | None = None) -> xr.DataArray:
    """Benjamini-Hochberg false discovery rate control (Wilks 2016 suggests alpha_FDR = 0.10).

    Cells with p_(i) <= (i / N) * alpha are significant, where N counts the cells
    tested. `within` restricts the test to a region (e.g. 54-90 N for the maps), so N
    is the number of cells in that region.
    """
    pv = p.values.copy()
    valid = np.isfinite(pv)
    if within is not None:
        valid &= within.values.astype(bool)
    flat = pv[valid]
    sig = np.zeros_like(pv, dtype=bool)
    if flat.size:
        order = np.argsort(flat)
        ranked = flat[order]
        thresh = alpha * np.arange(1, flat.size + 1) / flat.size
        below = np.nonzero(ranked <= thresh)[0]
        if below.size:
            cutoff = ranked[below.max()]
            sig[valid] = flat <= cutoff
    return xr.DataArray(sig, coords=p.coords, dims=p.dims, name="fdr_significant")


### Compound mapping and chemical constants
Copied from the supplied `megan.py`. The cited model source and case namelists are not attached, so verify these against the production cases before interpreting the chemical weighting. Carbon is emitted carbon mass; `koh` is a rate-weighted emissions index, not measured atmospheric OH reactivity.


In [ ]:
MEGAN_SPECIFIER = """
'ISOP = isoprene',
'MTERP = pinene_a + carene_3 + thujene_a + 2met_styrene + cymene_p + cymene_o + terpinolene + bornene + fenchene_a + ocimene_al + pinene_b + sabinene + camphene + limonene + phellandrene_a + terpinene_g + terpinene_a + phellandrene_b +  myrcene + ocimene_t_b + ocimene_c_b',
'BCARY = caryophyllene_b + bergamotene_a + bisabolene_b + farnescene_b + humulene_a',
'CH3OH = methanol', 'C2H5OH = ethanol', 'CH2O = formaldehyde',
'CH3CHO = acetaldehyde', 'CH3COOH = acetic_acid', 'CH3COCH3 = acetone',
'HCOOH = formic_acid', 'HCN = hydrogen_cyanide', 'CO = carbon_monoxide',
'C2H6 = ethane', 'C2H4 = ethene', 'C3H8 = propane', 'C3H6 = propene',
'BIGALK = pentane + hexane + heptane + tricyclene', 'BIGENE = butene',
'TOLUENE = toluene'
"""


FORMULAS = {
    "isoprene": "C5H8",
    # monoterpenes (MTERP)
    "pinene_a": "C10H16", "carene_3": "C10H16", "thujene_a": "C10H16", "2met_styrene": "C9H10",
    "cymene_p": "C10H14", "cymene_o": "C10H14", "terpinolene": "C10H16", "bornene": "C10H16",
    "fenchene_a": "C10H16", "ocimene_al": "C10H16", "pinene_b": "C10H16", "sabinene": "C10H16",
    "camphene": "C10H16", "limonene": "C10H16", "phellandrene_a": "C10H16", "terpinene_g": "C10H16",
    "terpinene_a": "C10H16", "phellandrene_b": "C10H16", "myrcene": "C10H16", "ocimene_t_b": "C10H16",
    "ocimene_c_b": "C10H16",
    # sesquiterpenes (BCARY)
    "caryophyllene_b": "C15H24", "bergamotene_a": "C15H24", "bisabolene_b": "C15H24",
    "farnescene_b": "C15H24", "humulene_a": "C15H24",
    # oxygenated
    "methanol": "CH4O", "ethanol": "C2H6O", "formaldehyde": "CH2O", "acetaldehyde": "C2H4O",
    "acetic_acid": "C2H4O2", "acetone": "C3H6O", "formic_acid": "CH2O2",
    # others
    "hydrogen_cyanide": "HCN", "carbon_monoxide": "CO",
    "ethane": "C2H6", "ethene": "C2H4", "propane": "C3H8", "propene": "C3H6",
    "pentane": "C5H12", "hexane": "C6H14", "heptane": "C7H16", "tricyclene": "C10H16",
    "butene": "C4H8", "toluene": "C7H8",
}


ATOMIC_MASS = {"C": 12.011, "H": 1.008, "O": 15.999, "N": 14.007}


GROUPS = {
    "ISOP+MTERP (prescribed in fBVOC)": ["ISOP", "MTERP"],
    "BCARY": ["BCARY"],
    "oxygenated VOC": ["CH3OH", "C2H5OH", "CH2O", "CH3CHO", "CH3COOH", "CH3COCH3", "HCOOH"],
    "CO": ["CO"],
    "other hydrocarbons": ["C2H6", "C2H4", "C3H8", "C3H6", "BIGALK", "BIGENE", "TOLUENE"],
    "HCN": ["HCN"],
}


PRESCRIBED_GROUP = "ISOP+MTERP (prescribed in fBVOC)"


SOA_PRECURSORS = {"ISOP", "MTERP"}


def parse_formula(formula: str) -> dict[str, int]:
    counts: dict[str, int] = {}
    for el, n in re.findall(r"([A-Z][a-z]?)(\d*)", formula):
        counts[el] = counts.get(el, 0) + (int(n) if n else 1)
    return counts


def molar_mass(formula: str) -> float:
    """g mol-1"""
    return sum(ATOMIC_MASS[el] * n for el, n in parse_formula(formula).items())


def carbon_number(formula: str) -> int:
    return parse_formula(formula).get("C", 0)


def parse_specifier(spec: str = MEGAN_SPECIFIER) -> dict[str, list[tuple[float, str]]]:
    """'MTERP = a + b' -> {'MTERP': [(1.0, 'a'), (1.0, 'b')]}. Coefficients like '0.5*x' are kept."""
    out: dict[str, list[tuple[float, str]]] = {}
    for entry in re.findall(r"'([^']+)'", spec):
        lhs, rhs = (s.strip() for s in entry.split("=", 1))
        terms = []
        for t in rhs.split("+"):
            t = t.strip()
            if "*" in t:
                c, name = t.split("*", 1)
                terms.append((float(c), name.strip()))
            else:
                terms.append((1.0, t))
        out[lhs] = terms
    return out


def compound_table(spec: str = MEGAN_SPECIFIER) -> list[dict]:
    """One row per MEGAN compound: CAM species, coefficient, MW, carbon number."""
    rows = []
    for cam, terms in parse_specifier(spec).items():
        for coeff, name in terms:
            if name not in FORMULAS:
                raise KeyError(f"No formula for MEGAN compound '{name}'; add it to FORMULAS")
            f = FORMULAS[name]
            rows.append({"compound": name, "cam_species": cam, "coeff": coeff, "formula": f,
                         "mw": molar_mass(f), "nC": carbon_number(f)})
    return rows


def group_of(cam_species: str) -> str:
    for g, members in GROUPS.items():
        if cam_species in members:
            return g
    return "unassigned"


K_BOLTZ = 1.380649e-23


def air_number_density(T: float, p_pa: float) -> float:
    """molecules cm-3"""
    return p_pa / (K_BOLTZ * T) * 1e-6


def _arr(A, B=0.0):
    return lambda T, M: A * math.exp(B / T)


def _troe(ko0, no, kinf0, ninf, fc):
    """Termolecular (+M) form as in mo_jpl.F90; the rate is multiplied by M because the
    reaction carries '+ M' in chem_mech.in (b, standard MOZART preprocessor convention)."""
    def k(T, M):
        ko = ko0 * (300.0 / T) ** no
        kinf = kinf0 * (300.0 / T) ** ninf
        x = ko * M / kinf
        return M * ko / (1 + x) * fc ** (1.0 / (1.0 + math.log10(x) ** 2))
    return k


def _co_oh(T, M):
    """CO + OH: CO_OH_M (troe, line 873) + usr_CO_OH_b (mo_usrrxt.F90)."""
    k1 = _troe(5.9e-33, 1.0, 1.1e-12, -1.3, 0.6)(T, M)
    kinf = 2.1e9 * (T / 300.0) ** 6.1
    ko = 1.5e-13
    x = ko * M / kinf
    k2 = ko / (1 + x) * 0.6 ** (1.0 / (1.0 + math.log10(x) ** 2))
    return k1 + k2


K_OH = {  # chem_mech.in line numbers in pp_tropstrat_mam_oslo
    "ISOP": _arr(2.54e-11, 410),            # 1006
    "MTERP": _arr(1.2e-11, 440),            # 1070
    "BCARY": _arr(2e-10),                   # 1067
    "CH3OH": _arr(2.9e-12, -345),           # 870
    "C2H5OH": _arr(6.9e-12, -230),          # 895
    "CH2O": _arr(5.5e-12, 125),             # 865
    "CH3CHO": _arr(4.63e-12, 350),          # 900
    "CH3COOH": _arr(7e-13),                 # 906
    "CH3COCH3": lambda T, M: 3.82e-11 * math.exp(-2000 / T) + 1.33e-13,   # usr_CH3COCH3_OH
    "HCOOH": _arr(4e-13),                   # 875
    "HCN": _troe(4.28e-33, 0.0, 9.3e-15, -4.42, 0.8),   # 874
    "CO": _co_oh,                           # 873 + usr_CO_OH_b
    "C2H6": _arr(7.66e-12, -1020),          # 898
    "C2H4": _troe(8.6e-29, 3.1, 9e-12, 0.85, 0.48),     # 915
    "C3H8": _arr(8.7e-12, -615),            # 927
    "C3H6": _troe(8e-27, 3.5, 3e-11, 0.0, 0.5),         # 939
    "BIGALK": _arr(3.5e-12),                # 981
    "BIGENE": _arr(5.4e-11),                # 945
    "TOLUENE": _arr(1.7e-12, 352),          # 1052
}


def k_oh(species: str, T: float = 298.0, p_pa: float = 101325.0) -> float:
    return K_OH[species](T, air_number_density(T, p_pa))


### Convert fluxes to budgets and compare members
A complete year is day-weighted, then multiplied by 365 days. CLM fluxes are assumed to be per land area: `flux × area × landfrac`. Missing requested fields stop execution rather than silently shrinking the budget.


In [ ]:
log = logging.getLogger(__name__)


SECONDS_PER_YEAR = 365 * 86400.0          # noleap calendar


WEIGHTINGS = {
    "mass": "Tg yr-1",
    "carbon": "Tg C yr-1",
    "koh": "Tmol(CO-eq) yr-1",
}




def meg_variables(spec: str = MEGAN_SPECIFIER) -> list[str]:
    return [f"MEG_{r['compound']}" for r in compound_table(spec)]


def species_fluxes(ds: xr.Dataset, spec: str = MEGAN_SPECIFIER, T_ref: float = 298.0) -> dict[str, xr.Dataset]:
    """Per-grid-cell fluxes per CAM species in the three weightings (per second, not per year).

    Returns {'mass': kg s-1, 'carbon': kg C s-1, 'koh': mol(CO-eq) s-1}, each a Dataset with one
    variable per CAM species, dims (time, lat, lon).
    """
    land_m2 = (ds["area"] * 1e6 * ds["landfrac"]).fillna(0.0)
    if "time" in land_m2.dims:
        land_m2 = land_m2.isel(time=0, drop=True)
    k_co = k_oh("CO", T_ref)
    out = {w: {} for w in WEIGHTINGS}
    for r in compound_table(spec):
        v = f"MEG_{r['compound']}"
        if v not in ds:
            continue
        kg_s = ds[v].fillna(0.0) * land_m2 * r["coeff"]
        mol_s = kg_s / (r["mw"] * 1e-3)
        s = r["cam_species"]
        add = {
            "mass": kg_s,
            "carbon": mol_s * r["nC"] * ATOMIC_MASS["C"] * 1e-3,
            "koh": mol_s * k_oh(s, T_ref) / k_co,
        }
        for w in WEIGHTINGS:
            out[w][s] = out[w][s] + add[w] if s in out[w] else add[w]
    return {w: xr.Dataset(d) for w, d in out.items()}


def load_annual_meg(pattern: str, spec: str = MEGAN_SPECIFIER, spinup_months: int = 12) -> xr.Dataset:
    """Annual means (month-length weighted) of all MEG_* fields of one CLM case, in memory.

    Everything downstream works on these (year, lat, lon) arrays: fluxes are linear in
    MEG_*, so annual totals of the weighted sums equal weighted sums of annual means.
    """
    ds = open_cam_monthly(pattern, meg_variables(spec), spinup_months, strict=True)
    for v in ("area", "landfrac"):
        if v not in ds:
            raise KeyError(f"{ds.attrs['case']}: CLM '{v}' missing; is this a CLM h0 file?")
    if ds.time.dt.calendar not in ("noleap", "365_day"):
        raise ValueError("This budget uses 365-day years; adapt integration for another calendar.")
    meg = [v for v in ds.data_vars if v.startswith("MEG_")]
    if "km" not in ds.area.attrs.get("units", "").lower():
        raise ValueError("Verify CLM area units are km2 before converting to m2.")
    for v in meg:
        units = ds[v].attrs.get("units", "").lower().replace(" ", "")
        if units not in {"kg/m2/s", "kg/m2/sec", "kgm-2s-1", "kgm^-2s^-1", "kgm**-2s**-1"}:  # CLM writes kg/m2/sec
            raise ValueError(f"{v}: verify kg compound m-2 s-1 units; found {units!r}")

    ann = annual_mean(ds[meg], ds["days_in_interval"]).load()
    for v in ("area", "landfrac"):
        a = ds[v]
        ann[v] = (a.isel(time=0, drop=True) if "time" in a.dims else a).load()
    for v in meg:
        if bool((ann[v].isnull() & (ann.landfrac > 0)).any()):
            raise ValueError(f"{v}: missing values on land in retained years")
    ann.attrs.update(case=ds.attrs["case"], missing=ds.attrs.get("missing", ""))
    ds.close()
    return ann


def annual_totals(ann: xr.Dataset, regions: dict[str, xr.DataArray], spec: str = MEGAN_SPECIFIER,
                  T_ref: float = 298.0) -> pd.DataFrame:
    """Annual regional totals per CAM species and weighting, in WEIGHTINGS units.
    `ann` = output of load_annual_meg (dims year, lat, lon)."""
    flux = species_fluxes(ann, spec, T_ref)
    scale = {"mass": 1e-9, "carbon": 1e-9, "koh": 1e-12}     # kg -> Tg ; mol -> Tmol
    rows = []
    for w, fds in flux.items():
        for rname, mask in regions.items():
            tot = fds.where(mask, 0.0).sum(("lat", "lon")) * SECONDS_PER_YEAR * scale[w]
            df = tot.to_dataframe().reset_index().melt(id_vars="year", var_name="species", value_name="value")
            df["region"], df["weighting"] = rname, w
            rows.append(df)
    out = pd.concat(rows, ignore_index=True)
    out["group"] = out["species"].map(group_of)
    out["case"] = ann.attrs.get("case", "")
    return out


def _regions(ds: xr.Dataset, modified_path: str) -> dict[str, xr.DataArray]:
    mod = load_modified_cells(modified_path, ds)
    return {
        "global": xr.ones_like(mod, dtype=bool),
        "modified_cells": mod,
        "NH_45_90": band_mask(ds, 45, 90),
        "tropics": band_mask(ds, -30, 30),
    }


def _group_totals(totals: pd.DataFrame) -> pd.DataFrame:
    g = totals.groupby(["member", "region", "weighting", "group", "year"], as_index=False)["value"].sum()
    others = (g[g.group != PRESCRIBED_GROUP]
              .groupby(["member", "region", "weighting", "year"], as_index=False)["value"].sum())
    others["group"] = "ALL OTHERS"
    allg = g.groupby(["member", "region", "weighting", "year"], as_index=False)["value"].sum()
    allg["group"] = "ALL MEGAN"
    return pd.concat([g, others, allg], ignore_index=True)


def q1_table(gt: pd.DataFrame) -> pd.DataFrame:
    """Mean annual total per group and its share of all MEGAN emissions (%)."""
    m = gt.groupby(["member", "region", "weighting", "group"], as_index=False)["value"].mean()
    tot = m[m.group == "ALL MEGAN"].set_index(["member", "region", "weighting"])["value"]
    m["share_pct"] = 100 * m["value"] / m.set_index(["member", "region", "weighting"]).index.map(tot)
    m["units"] = m["weighting"].map(WEIGHTINGS)
    return m


def gross_share(totals: pd.DataFrame, contrasts: list[EmissionContrast]) -> pd.DataFrame:
    """Share of the others in the GROSS change, sum over species of |mean Delta E|.

    Robust to cancellation: if isoprene decreases while monoterpenes increase, the net
    Delta E(ISOP+MTERP) can be ~0 and the ratio others/ISOP+MTERP meaningless, but the
    gross change is not. Species level = CAM species (19).
    """
    rows = []
    m = totals.groupby(["member", "region", "weighting", "species", "group"], as_index=False)["value"].mean()
    for c in contrasts:
        a = m[m.member == c.name + "::minuend"]
        b = m[m.member == c.name + "::subtrahend"]
        mg = a.merge(b, on=["region", "weighting", "species", "group"], suffixes=("_a", "_b"))
        mg["absd"] = (mg["value_a"] - mg["value_b"]).abs()
        for (reg, w), g in mg.groupby(["region", "weighting"]):
            tot = g["absd"].sum()
            oth = g.loc[g.group != PRESCRIBED_GROUP, "absd"].sum()
            rows.append({"contrast": c.name, "region": reg, "weighting": w,
                         "others_share_of_gross_change": oth / tot if tot else np.nan})
    return pd.DataFrame(rows)


def q2_tables(gt: pd.DataFrame, contrasts: list[EmissionContrast]) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows, summ = [], []
    for c in contrasts:
        a = gt[gt.member == c.name + "::minuend"]
        b = gt[gt.member == c.name + "::subtrahend"]
        mg = a.merge(b, on=["region", "weighting", "group", "year"], suffixes=("_a", "_b"))
        mg["delta"] = mg["value_a"] - mg["value_b"]
        for (reg, w, grp), g in mg.groupby(["region", "weighting", "group"]):
            s = paired_summary(g["delta"].to_numpy())
            rows.append({"contrast": c.name, "region": reg, "weighting": w, "group": grp,
                         "E_minuend": g["value_a"].mean(), "E_subtrahend": g["value_b"].mean(),
                         "delta": s["mean"], "ci_lo": s["ci_lo"], "ci_hi": s["ci_hi"], "p": s["p"],
                         "rel_change_pct": 100 * s["mean"] / g["value_b"].mean() if g["value_b"].mean() else np.nan,
                         "units": WEIGHTINGS[w], "n_years": s["n"]})
    q2 = pd.DataFrame(rows)
    for (cn, reg, w), g in q2.groupby(["contrast", "region", "weighting"]):
        gi = g.set_index("group")
        d_pre = gi.loc[PRESCRIBED_GROUP, "delta"]
        d_oth = gi.loc["ALL OTHERS", "delta"]
        e_oth = gi.loc["ALL OTHERS", "E_subtrahend"]
        pre_sig = bool(gi.loc[PRESCRIBED_GROUP, "ci_lo"] > 0 or gi.loc[PRESCRIBED_GROUP, "ci_hi"] < 0)
        summ.append({
            "contrast": cn, "region": reg, "weighting": w, "units": WEIGHTINGS[w],
            "dE_ISOP_MTERP": d_pre, "dE_others": d_oth,
            "ISOP_MTERP_ci_excludes_0": pre_sig,
            # only meaningful when the ISOP+MTERP change itself is distinguishable from 0
            "others_over_ISOP_MTERP": d_oth / d_pre if (pre_sig and d_pre) else np.nan,
            # fraction of the others' emissions changed by LCC: scales the D1 artifact (b)
            "f_others_changed": d_oth / gi.loc["ALL OTHERS", "E_subtrahend"] if e_oth else np.nan,
            "others_ci_excludes_0": bool(gi.loc["ALL OTHERS", "ci_lo"] > 0 or gi.loc["ALL OTHERS", "ci_hi"] < 0),
        })
    return q2, pd.DataFrame(summ)


def delta_carbon_maps(a: xr.Dataset, b: xr.Dataset, spec: str = MEGAN_SPECIFIER) -> xr.Dataset:
    """Time-mean Delta carbon flux, g C m-2 (grid-cell area) yr-1, for ISOP+MTERP and the others.
    a, b = outputs of load_annual_meg."""
    out = {}
    years = np.intersect1d(a["year"].values, b["year"].values)
    for ds, key in ((a, "a"), (b, "b")):
        f = species_fluxes(ds.sel(year=years), spec)["carbon"]
        cell_m2 = ds["area"] * 1e6
        ann = f.mean("year")
        pre = sum(ann[s] for s in GROUPS[PRESCRIBED_GROUP] if s in ann)
        oth = sum(ann[s] for s in ann.data_vars if s not in GROUPS[PRESCRIBED_GROUP])
        out[key] = xr.Dataset({"ISOP+MTERP": pre, "others": oth}) / cell_m2 * 1e3 * SECONDS_PER_YEAR
    d = (out["a"] - out["b"]).load()
    for v in d.data_vars:
        d[v].attrs["units"] = "g C m-2 yr-1"
    return d


### Plotting helpers


In [ ]:
HAS_CARTOPY = False  # Set True only if Cartopy AND its coastline data are installed.
if HAS_CARTOPY:
    import cartopy.crs as ccrs
INK, INK2, GRID = "#0b0b0b", "#52514e", "#d9d8d4"
def _coastlines_available():
    # Never trigger a network download on a compute node.
    if not HAS_CARTOPY:
        return False
    import cartopy
    from pathlib import Path
    rel = Path("shapefiles/natural_earth/physical/ne_110m_coastline.shp")
    return any((Path(cartopy.config[k]) / rel).is_file() for k in ("data_dir", "pre_existing_data_dir"))

if HAS_CARTOPY:
    import cartopy.crs as ccrs


REGION_COLOR = {"modified_cells": "#2a78d6", "global": "#8a8984", "NH_45_90": "#6250d6", "tropics": "#eb6834"}


CONTRAST_COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]     # fixed order: slots 1-3


GROUP_ORDER = list(GROUPS) + ["ALL OTHERS"]


WEIGHT_TITLE = {"mass": "mass", "carbon": "emitted carbon",
                "koh": "rate-weighted emissions at T_ref"}


def _style(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(INK2)
    ax.tick_params(colors=INK2, labelsize=8)
    ax.grid(axis="x", color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    ax.axvline(0, color=INK2, lw=0.8)


def plot_q1_shares(q1: pd.DataFrame, member: str, regions=("global", "modified_cells")) -> plt.Figure:
    groups = list(GROUPS)
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True, constrained_layout=True)
    y = np.arange(len(groups))
    h = 0.8 / len(regions)
    for ax, w in zip(axes, WEIGHTINGS):
        sub = q1[(q1.member == member) & (q1.weighting == w)]
        for k, r in enumerate(regions):
            s = sub[sub.region == r].set_index("group").reindex(groups)["share_pct"]
            ys = y - 0.4 + h * (k + 0.5)
            ax.barh(ys, s, h * 0.9, color=REGION_COLOR[r], label=r.replace("_", " "))
            for yy, v in zip(ys, s):
                if np.isfinite(v):
                    ax.text(v + 1, yy, f"{v:.1f}%", va="center", fontsize=7, color=INK2)
        _style(ax)
        ax.set_xlim(0, 115)
        ax.set_title(WEIGHT_TITLE[w], fontsize=9, color=INK)
        ax.set_xlabel("share of all MEGAN emissions (%)", fontsize=8, color=INK2)
    axes[0].set_yticks(y, groups, fontsize=8, color=INK)
    axes[0].invert_yaxis()
    axes[0].legend(frameon=False, fontsize=8, loc="lower right")
    fig.suptitle(f"Q1 - composition of MEGAN emissions in {member}", fontsize=10, color=INK)
    return fig


def plot_q2_deltas(q2: pd.DataFrame, contrasts: list[str], region: str = "global") -> plt.Figure:
    fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.8), sharey=True, constrained_layout=True)
    y = np.arange(len(GROUP_ORDER))
    h = 0.8 / len(contrasts)
    for ax, w in zip(axes, WEIGHTINGS):
        sub = q2[(q2.region == region) & (q2.weighting == w)]
        for k, cn in enumerate(contrasts):
            s = sub[sub.contrast == cn].set_index("group").reindex(GROUP_ORDER)
            ys = y - 0.4 + h * (k + 0.5)
            ax.barh(ys, s["delta"], h * 0.9, color=CONTRAST_COLORS[k % 3], label=cn)
            err = np.vstack([s["delta"] - s["ci_lo"], s["ci_hi"] - s["delta"]])
            ax.errorbar(s["delta"], ys, xerr=err, fmt="none", ecolor=INK2, elinewidth=0.9, capsize=2)
        _style(ax)
        ax.set_title(WEIGHT_TITLE[w], fontsize=9, color=INK)
        ax.set_xlabel(f"$\\Delta$E ({WEIGHTINGS[w]})", fontsize=8, color=INK2)
    axes[0].set_yticks(y, GROUP_ORDER, fontsize=8, color=INK)
    axes[0].invert_yaxis()
    for ax in axes:
        ax.axhline(len(GROUPS) - 0.5, color=GRID, lw=1)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside upper center", ncol=len(contrasts), frameon=False, fontsize=8)
    fig.suptitle(f"Q2 - change in MEGAN emissions, {region.replace('_', ' ')} "
                 "(bars: mean, whiskers: 95% interval over years)", fontsize=9, color=INK2, y=-0.04)
    return fig


def plot_delta_maps(maps: dict[str, xr.Dataset], modified: xr.DataArray | None = None,
                    lat_min: float = 45, vmax_quantile: float = 0.99) -> plt.Figure:
    names = list(maps)
    cols = ["ISOP+MTERP", "others"]
    fig = plt.figure(figsize=(8.2, 3.6 * len(names)), constrained_layout=True)
    axes = np.empty((len(names), 2), dtype=object)
    for i in range(len(names)):
        for j in range(2):
            kw = {"projection": ccrs.NorthPolarStereo()} if HAS_CARTOPY else {}
            axes[i, j] = fig.add_subplot(len(names), 2, 2 * i + j + 1, **kw)
    tr = {"transform": ccrs.PlateCarree()} if HAS_CARTOPY else {}
    # One colour scale PER PANEL: the others are much smaller than ISOP+MTERP, and the
    # LCC + climate-change contrast is much larger than the LCC-only ones.
    for j, col in enumerate(cols):
        for i, n in enumerate(names):
            vals = np.abs(maps[n][col].sel(lat=slice(lat_min, 90)).values).ravel()
            vmax = float(np.nanquantile(vals, vmax_quantile)) or 1e-6
            norm = TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax)
            ax = axes[i, j]
            da = maps[n][col].sel(lat=slice(lat_min - 2, 90))
            lon = da["lon"].values
            da_c = xr.concat([da, da.isel(lon=0).assign_coords(lon=lon[0] + 360)], dim="lon")
            m = ax.pcolormesh(da_c["lon"], da_c["lat"], da_c, cmap="RdBu_r", norm=norm, shading="nearest", **tr)
            if modified is not None:
                md = modified.sel(lat=slice(lat_min - 2, 90)).astype(float)
                ax.contour(md["lon"], md["lat"], md, levels=[0.5], colors=INK, linewidths=0.5, **tr)
            if HAS_CARTOPY:
                ax.set_extent([-180, 180, lat_min, 90], ccrs.PlateCarree())
                if _coastlines_available():
                    ax.coastlines(lw=0.3, color=INK2)
            if not HAS_CARTOPY:
                ax.set(xlim=(0, 360), ylim=(lat_min, 90), xlabel="longitude (degrees east)", ylabel="latitude (degrees north)")
            ax.set_title(f"{n}: $\\Delta$E carbon, {col}", fontsize=9, color=INK)
            cb = fig.colorbar(m, ax=ax, orientation="vertical", shrink=0.8, pad=0.02, extend="both")
            cb.set_label("g C m$^{-2}$ yr$^{-1}$", fontsize=7, color=INK2)
            cb.ax.tick_params(labelsize=7, colors=INK2)
    fig.suptitle("$\\Delta$E carbon (grid-cell mean). Note: each panel has its own colour scale",
                 fontsize=9, color=INK2)
    return fig


## 2. Load each interactive member once
All requested MEG fields must exist in every input file. Missing land fluxes stop execution. This retains the original noleap-calendar budget convention. Inputs must already be on the same grid; no regridding is performed.


In [ ]:
annual = {alias: load_annual_meg(pattern, spinup_months=SPINUP_MONTHS) for alias, pattern in CLM.items()}
reference = annual["CTRL_PD"]
for alias, ds in annual.items():
    xr.align(reference, ds, join="exact")
    if ds.sizes["year"] < 3:
        raise ValueError(f"{alias}: need at least three complete paired years")
    xr.testing.assert_allclose(reference.area, ds.area)
    xr.testing.assert_allclose(reference.landfrac, ds.landfrac)
    print(alias, ds.year.values)
regions = _regions(reference, str(MODIFIED_CELLS))
regions["unmodified_cells"] = ~regions["modified_cells"]


## 3. Calculate annual regional budgets once per member
The four configurations share the same retained years. Reusing these small budget tables avoids recomputing all species and regions for each contrast.


In [ ]:
budget = {}
for alias, ds in annual.items():
    budget[alias] = annual_totals(ds, regions, T_ref=T_REF)
    budget[alias]["member"] = alias
alias_for_path = {path: alias for alias, path in CLM.items()}
frames = [budget[alias] for alias in MEMBERS_Q1]
for contrast in CONTRASTS:
    for role, path in (("minuend", contrast.minuend), ("subtrahend", contrast.subtrahend)):
        frames.append(budget[alias_for_path[path]].assign(member=f"{contrast.name}::{role}"))
totals = pd.concat(frames, ignore_index=True)
grouped = _group_totals(totals)
q1 = q1_table(grouped[~grouped.member.str.contains("::")])
q2, summary = q2_tables(grouped, CONTRASTS)
summary = summary.merge(gross_share(totals, CONTRASTS), on=["contrast", "region", "weighting"], validate="one_to_one")
for name, data in [("totals", totals), ("q1", q1), ("q2", q2), ("q2_summary", summary)]:
    data.to_csv(TABLE_DIR / f"D2_{name}.csv", index=False)


## 4. How large are emissions outside ISOP and MTERP?
Q1 shows CTRL totals and shares. A large share establishes that substantial emitted material is outside the prescribed pair; it does not establish a proportionate radiation effect. These are budgets of the compounds listed in the supplied specifier, not every possible biogenic emission pathway.


In [ ]:
display(q1.pivot_table(index=["member", "region", "group"], columns="weighting", values=["value", "share_pct"]).round(3))
for member in MEMBERS_Q1:
    fig = plot_q1_shares(q1, member)
    fig.savefig(FIG_DIR / f"D2_Q1_shares_{member}.png", dpi=200, bbox_inches="tight")
    plt.show()


## 5. What changes between simulations?
- `LCC-CTRL_PD`: forest-expansion comparison in PD.
- `LCC-CTRL_FUT`: forest-expansion comparison in FUT.
- `CTRL_FUT-CTRL_PD`: background environmental change with control land cover.
- `LCC_FUT-CTRL_PD`: their combined emissions difference.

These labels describe the intended design; verify that each pair differs only as intended. The combined emissions difference equals the sum of the FUT LCC and background differences by subtraction. That identity is not a linearity assumption about atmospheric responses.

Read signed changes first. `others_share_of_gross_change` adds absolute changes across CAM species after spatial integration and time averaging; it avoids cancellation between species, but not cancellation across grid cells. `others_over_ISOP_MTERP` can be unstable if the denominator is small. `f_others_changed` is an emissions fraction, not a radiative scaling factor.


In [ ]:
cols = ["dE_ISOP_MTERP", "dE_others", "others_share_of_gross_change", "others_over_ISOP_MTERP", "f_others_changed"]
display(summary.set_index(["contrast", "region", "weighting"])[cols].round(4))
keys = ["region", "weighting", "group"]
pivot = q2.pivot(index=keys, columns="contrast", values="delta")
np.testing.assert_allclose(pivot["LCC_FUT-CTRL_PD"], pivot["LCC-CTRL_FUT"] + pivot["CTRL_FUT-CTRL_PD"], rtol=1e-10, atol=1e-10)
GROUPS_OF_CONTRASTS = {"LCC_only": ["LCC-CTRL_PD", "LCC-CTRL_FUT"], "background_and_combined": ["CTRL_FUT-CTRL_PD", "LCC_FUT-CTRL_PD"]}
for group_name, names in GROUPS_OF_CONTRASTS.items():
    for region in ("global", "modified_cells", "unmodified_cells"):
        fig = plot_q2_deltas(q2, names, region=region)
        fig.savefig(FIG_DIR / f"D2_Q2_{group_name}_{region}.png", dpi=200, bbox_inches="tight")
        plt.show()


## 6. Where do carbon emissions change?
Maps show carbon per total grid-cell area, not per land area. Each panel has its own scale: compare numbers on colour bars, not colour intensity between panels. The northern maps describe the forest-expansion region; use the global/unmodified budget rows above to assess changes elsewhere.


In [ ]:
maps = {c.name: delta_carbon_maps(annual[alias_for_path[c.minuend]], annual[alias_for_path[c.subtrahend]]) for c in CONTRASTS}
fig = plot_delta_maps(maps, modified=regions["modified_cells"])
fig.savefig(FIG_DIR / "D2_delta_maps.png", dpi=200, bbox_inches="tight")
plt.show()


## 7. Optional CAM versus CLM check
Compare the moles emitted by CLM with CAM MEG_ISOP/MEG_MTERP over the same years. This checks the assumed coupling conventions. It does not compare against SFISOP/SFMTERP or independently verify other sources. Confirm CAM fields are kg of CAM species per grid-cell m² per second before enabling.


In [ ]:
if RUN_CAM_CROSSCHECK:
    cam = open_cam_monthly(CAM_CTRL_PD, ["MEG_ISOP", "MEG_MTERP"], SPINUP_MONTHS)
    cam_ann = annual_mean(cam[["MEG_ISOP", "MEG_MTERP"]], cam.days_in_interval).load()
    np.testing.assert_array_equal(cam_ann.year, reference.year)
    cell_m2 = area_weights(cam) * 4 * np.pi * 6.37122e6**2
    rows = []
    for species, formula in (("ISOP", "C5H8"), ("MTERP", "C10H16")):
        clm_mol = sum((reference[f"MEG_{r['compound']}"] * reference.area * 1e6 * reference.landfrac * r["coeff"] / (r["mw"] * 1e-3)).sum(("lat", "lon")) for r in compound_table() if r["cam_species"] == species)
        cam_mol = (cam_ann[f"MEG_{species}"] * cell_m2 / (molar_mass(formula) * 1e-3)).sum(("lat", "lon"))
        rows.append({"species": species, "CLM_Tmol_yr": float(clm_mol.mean()) * SECONDS_PER_YEAR * 1e-12, "CAM_Tmol_yr": float(cam_mol.mean()) * SECONDS_PER_YEAR * 1e-12})
    crosscheck = pd.DataFrame(rows)
    crosscheck["CAM/CLM"] = crosscheck.CAM_Tmol_yr / crosscheck.CLM_Tmol_yr
    display(crosscheck)
    crosscheck.to_csv(TABLE_DIR / "D2_CAM_CLM_crosscheck.csv", index=False)
    cam.close()
else:
    print("Optional CAM check disabled.")


## 8. What these budgets do and do not decide
D2 quantifies emitted material and how the chosen simulations differ. It cannot turn an emissions share into a share of forcing or methane-lifetime change. Carbon mass is not total OH demand, and rate-weighted emissions are not atmospheric OH reactivity.

Unexpected changes outside modified cells are a reason to inspect forcing, namelists, LAI, vegetation carbon and restart states. They do not identify a spin-up mismatch by themselves. The previous automatic estimate `emissions fraction × D1 far-field radiation` has been removed because it assigns a cause and response scaling that these diagnostics do not establish.

Before choosing extra simulations, confirm the intended experimental differences and the actual prescribed species. Additional matched control experiments can help attribution, but their differences must be interpreted as responses conditional on the chosen background.
